# Responsible and Safe AI – External Practical (Set D)
## Privacy-Preserving Training of a PIN-Code Digit Recogniser

**Dataset:** MNIST handwritten digits  
**Required setup:** CPU, 20,000 training images, batch size 256, 3 epochs, `torch.manual_seed(42)`, GroupNorm, `δ = 1e-5`, `max_grad_norm = 1.0`.

This notebook completes Tasks 1–6 and prints the four technical-question answers using the results produced by your own run.

> **Important:** The examination paper says the expected output is a format guide only. Do not copy its sample accuracies or privacy values as your measured results. Run every cell and use the values printed by this notebook in your report.

## Aim
To train a handwritten digit classifier on a 20,000-image MNIST subset, compare non-private and differentially private (DP-SGD) training, measure privacy expenditure, perform a simple membership-inference confidence check, and plot the privacy–utility trade-off.

## Algorithm
1. Set the random seed and select CPU.
2. Load MNIST and create a reproducible subset of 20,000 training images.
3. Build a small CNN using GroupNorm (not BatchNorm).
4. Train and evaluate a non-private baseline for 3 epochs.
5. Train separate DP models for target ε = 10, 5, and 1 using `PrivacyEngine.make_private_with_epsilon`.
6. Train another DP model with fixed noise multiplier 1.1; record ε after each epoch.
7. Compare mean maximum-softmax confidence on training and test images for the baseline and ε = 1 model.
8. Plot accuracy versus ε and ε versus epoch, tabulate results, and print report-ready answers.

In [ ]:
# If Opacus is not installed, uncomment and run this cell:
# %pip install opacus

import os
import copy
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

try:
    from opacus import PrivacyEngine
except ImportError as exc:
    raise ImportError("Opacus is required. Run: %pip install opacus") from exc

SEED = 42
BATCH_SIZE = 256
EPOCHS = 3
DELTA = 1e-5
MAX_GRAD_NORM = 1.0
TRAIN_SUBSET_SIZE = 20_000
DEVICE = torch.device("cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("Opacus imported successfully")

## 1. Load MNIST and prepare data

The training set is restricted to exactly 20,000 examples. The full 10,000-image MNIST test set is used for evaluation. The subset selection is reproducible.

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])

# download=True uses the local torchvision cache if present and downloads only if missing.
train_full = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

generator = torch.Generator().manual_seed(SEED)
subset_indices = torch.randperm(len(train_full), generator=generator)[:TRAIN_SUBSET_SIZE].tolist()
train_dataset = Subset(train_full, subset_indices)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=False
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=False
)

print(f"Training subset: {len(train_dataset):,} images")
print(f"Test set: {len(test_dataset):,} images")

## 2. Build the CNN with GroupNorm

GroupNorm is used because Opacus does not support BatchNorm in this DP-SGD setup. The model accepts one-channel 28×28 MNIST images and predicts 10 digit classes.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.GroupNorm(4, 16),
            nn.ReLU(),
            nn.MaxPool2d(2),                 # 28 -> 14

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.GroupNorm(8, 32),
            nn.ReLU(),
            nn.MaxPool2d(2),                 # 14 -> 7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def make_model():
    torch.manual_seed(SEED)
    return SmallCNN().to(DEVICE)


model_check = make_model()
print(model_check)

## 3. Helper functions: training, evaluation and confidence

The confidence measure is the mean of the maximum softmax probability. A larger train–test gap can indicate that model confidence differs between seen and unseen examples; it is only a simple diagnostic, not a complete membership-inference attack.

In [ ]:
criterion = nn.CrossEntropyLoss()

def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, 100.0 * total_correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        logits = model(images)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return 100.0 * correct / total


@torch.no_grad()
def mean_max_softmax_confidence(model, loader):
    model.eval()
    confidence_sum = 0.0
    count = 0
    for images, _ in loader:
        images = images.to(DEVICE)
        probabilities = torch.softmax(model(images), dim=1)
        confidence_sum += probabilities.max(dim=1).values.sum().item()
        count += images.size(0)
    return confidence_sum / count


def fresh_train_loader():
    # New deterministic shuffle generator for each experiment.
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                      generator=g, num_workers=0, pin_memory=False)

## Task 1 – Train the non-private baseline

Train the GroupNorm CNN for 3 epochs without a PrivacyEngine and record test accuracy.

In [ ]:
baseline_model = make_model()
baseline_optimizer = torch.optim.SGD(baseline_model.parameters(), lr=0.05, momentum=0.9)

baseline_history = []
for epoch in range(1, EPOCHS + 1):
    loss, train_acc = train_one_epoch(baseline_model, fresh_train_loader(), baseline_optimizer)
    test_acc = evaluate(baseline_model, test_loader)
    baseline_history.append({"epoch": epoch, "loss": loss,
                             "train_accuracy": train_acc, "test_accuracy": test_acc})
    print(f"Baseline | Epoch {epoch}/{EPOCHS} | loss={loss:.4f} | "
          f"train_acc={train_acc:.2f}% | test_acc={test_acc:.2f}%")

baseline_accuracy = evaluate(baseline_model, test_loader)
print(f"\nBaseline test accuracy: {baseline_accuracy:.2f}%")

## Task 2 – Train DP models for ε = 10, 5 and 1

Each target ε gets a freshly initialized model and optimizer. `make_private_with_epsilon` configures the noise multiplier for the requested target privacy budget over the specified number of epochs. The final ε is also measured using the accountant.

In [ ]:
dp_results = []
dp_models = {}

for target_epsilon in [10.0, 5.0, 1.0]:
    print(f"\nTraining DP model for target epsilon = {target_epsilon:g}")
    dp_model = make_model()
    optimizer = torch.optim.SGD(dp_model.parameters(), lr=0.05, momentum=0.9)
    privacy_engine = PrivacyEngine()

    private_model, private_optimizer, private_loader = privacy_engine.make_private_with_epsilon(
        module=dp_model,
        optimizer=optimizer,
        data_loader=fresh_train_loader(),
        target_epsilon=target_epsilon,
        target_delta=DELTA,
        epochs=EPOCHS,
        max_grad_norm=MAX_GRAD_NORM,
    )

    for epoch in range(1, EPOCHS + 1):
        loss, train_acc = train_one_epoch(private_model, private_loader, private_optimizer)
        print(f"  Epoch {epoch}/{EPOCHS} | loss={loss:.4f} | train_acc={train_acc:.2f}%")

    test_acc = evaluate(private_model, test_loader)
    epsilon_spent = privacy_engine.get_epsilon(delta=DELTA)

    dp_results.append({
        "target_epsilon": target_epsilon,
        "delta": DELTA,
        "clip_C": MAX_GRAD_NORM,
        "epochs": EPOCHS,
        "test_accuracy": test_acc,
        "epsilon_spent": epsilon_spent
    })
    dp_models[target_epsilon] = private_model
    print(f"  Test accuracy: {test_acc:.2f}% | epsilon spent: {epsilon_spent:.4f}")

dp_results_df = pd.DataFrame(dp_results)
print("\nDP accuracy results:")
display(dp_results_df.round(4))

## Task 3 – Fixed noise multiplier = 1.1; record ε after every epoch

This is a separate DP training run. The accountant accumulates privacy loss as training steps are performed, so ε is measured at the end of each epoch.

In [ ]:
fixed_noise = 1.1
fixed_model = make_model()
fixed_optimizer = torch.optim.SGD(fixed_model.parameters(), lr=0.05, momentum=0.9)
fixed_privacy_engine = PrivacyEngine()

fixed_private_model, fixed_private_optimizer, fixed_private_loader = fixed_privacy_engine.make_private(
    module=fixed_model,
    optimizer=fixed_optimizer,
    data_loader=fresh_train_loader(),
    noise_multiplier=fixed_noise,
    max_grad_norm=MAX_GRAD_NORM,
)

fixed_epsilon_history = []
for epoch in range(1, EPOCHS + 1):
    loss, train_acc = train_one_epoch(
        fixed_private_model, fixed_private_loader, fixed_private_optimizer
    )
    epsilon_now = fixed_privacy_engine.get_epsilon(delta=DELTA)
    fixed_epsilon_history.append({"epoch": epoch, "epsilon_spent": epsilon_now})
    print(f"Fixed noise={fixed_noise} | Epoch {epoch} | "
          f"loss={loss:.4f} | train_acc={train_acc:.2f}% | epsilon={epsilon_now:.4f}")

fixed_epsilon_df = pd.DataFrame(fixed_epsilon_history)
print("\nPrivacy expenditure after each epoch:")
display(fixed_epsilon_df.round(4))

## Task 4 – Simple membership-inference confidence check

For each model, calculate mean maximum-softmax confidence on the 20,000 training images and on the test images. The confidence gap is `train confidence − test confidence`.

In [ ]:
# Evaluate confidence on exactly the training subset used for training.
confidence_train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

baseline_train_conf = mean_max_softmax_confidence(baseline_model, confidence_train_loader)
baseline_test_conf = mean_max_softmax_confidence(baseline_model, test_loader)

dp1_model = dp_models[1.0]
dp1_train_conf = mean_max_softmax_confidence(dp1_model, confidence_train_loader)
dp1_test_conf = mean_max_softmax_confidence(dp1_model, test_loader)

membership_results = pd.DataFrame([
    {"model": "Baseline", "train_confidence": baseline_train_conf,
     "test_confidence": baseline_test_conf,
     "gap": baseline_train_conf - baseline_test_conf},
    {"model": "DP (target epsilon=1)", "train_confidence": dp1_train_conf,
     "test_confidence": dp1_test_conf,
     "gap": dp1_train_conf - dp1_test_conf}
])

print("Membership check (mean max-softmax confidence):")
display(membership_results.round(4))

## Task 5 – Plots

The first plot uses the measured baseline accuracy and the measured accuracy for each DP target. The second plot shows the measured ε after each epoch for the fixed-noise run.

In [ ]:
plot_eps = [0.0] + [row["target_epsilon"] for row in dp_results]
plot_acc = [baseline_accuracy] + [row["test_accuracy"] for row in dp_results]

# Sort by epsilon for a readable left-to-right privacy-budget axis.
order = np.argsort(plot_eps)
sorted_eps = np.array(plot_eps)[order]
sorted_acc = np.array(plot_acc)[order]

plt.figure(figsize=(7, 5))
plt.plot(sorted_eps, sorted_acc, marker="o")
plt.xlabel("Target ε (baseline shown at ε = 0)")
plt.ylabel("Test accuracy (%)")
plt.title("Privacy–Utility Curve: Test Accuracy vs ε")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("accuracy_vs_epsilon.png", dpi=150)
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(fixed_epsilon_df["epoch"], fixed_epsilon_df["epsilon_spent"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("ε spent")
plt.title("Privacy Expenditure vs Epoch (noise multiplier = 1.1)")
plt.xticks([1, 2, 3])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("epsilon_vs_epoch.png", dpi=150)
plt.show()

print("Saved plots: accuracy_vs_epsilon.png and epsilon_vs_epoch.png")

## Task 6 – Combined results and recommendation

The notebook recommends the smallest tested ε whose measured accuracy is within 1 percentage point of the baseline, if one exists. Otherwise, it recommends the tested ε with the highest accuracy subject to the privacy preference being explicit. Review the printed result and justify the final choice for the postal-service use case.

In [ ]:
combined_rows = [{
    "model": "Baseline (no DP)", "target_epsilon": np.nan,
    "delta": np.nan, "clip_C": np.nan, "epochs": EPOCHS,
    "test_accuracy": baseline_accuracy, "epsilon_spent": np.nan
}]
for row in dp_results:
    combined_rows.append({
        "model": f"DP (target ε={row['target_epsilon']:g})",
        "target_epsilon": row["target_epsilon"],
        "delta": row["delta"], "clip_C": row["clip_C"],
        "epochs": row["epochs"], "test_accuracy": row["test_accuracy"],
        "epsilon_spent": row["epsilon_spent"]
    })

results_table = pd.DataFrame(combined_rows)
display(results_table.round(4))

accuracy_cost_eps1 = baseline_accuracy - dp_results_df.loc[
    dp_results_df["target_epsilon"] == 1.0, "test_accuracy"
].iloc[0]

# Operational recommendation rule: strongest privacy among tested settings
# that loses no more than 1 percentage point versus the baseline.
eligible = dp_results_df[
    dp_results_df["test_accuracy"] >= baseline_accuracy - 1.0
].sort_values("target_epsilon")

if len(eligible):
    recommended_epsilon = float(eligible.iloc[0]["target_epsilon"])
    recommendation_reason = (
        "It is the smallest tested ε whose measured accuracy is no more than "
        "1 percentage point below the non-private baseline."
    )
else:
    best_row = dp_results_df.sort_values("test_accuracy", ascending=False).iloc[0]
    recommended_epsilon = float(best_row["target_epsilon"])
    recommendation_reason = (
        "No tested DP setting stayed within 1 percentage point of baseline; "
        "this setting had the highest measured DP accuracy, so the privacy/utility "
        "trade-off should be reviewed with the service's risk requirements."
    )

print(f"Baseline accuracy: {baseline_accuracy:.2f}%")
print(f"Accuracy cost at target ε=1: {accuracy_cost_eps1:.2f} percentage points")
print(f"Recommended tested ε: {recommended_epsilon:g}")
print("Reason:", recommendation_reason)

## Technical Questions – Report-ready answers (Q1–Q4)

Run this cell after all tasks. It generates answers from the measured values above. Copy these answers and your tables/plots into the PDF report, editing the wording if required.

In [ ]:
acc_by_eps = {
    float(row["target_epsilon"]): float(row["test_accuracy"])
    for row in dp_results
}
eps_epoch_values = fixed_epsilon_df["epsilon_spent"].tolist()

gap_baseline = float(membership_results.loc[
    membership_results["model"] == "Baseline", "gap"
].iloc[0])
gap_dp1 = float(membership_results.loc[
    membership_results["model"] == "DP (target epsilon=1)", "gap"
].iloc[0])

print("Q1. Baseline and DP accuracy")
print(f"Baseline test accuracy = {baseline_accuracy:.2f}%.")
print(f"DP test accuracy: ε=10: {acc_by_eps[10.0]:.2f}%, "
      f"ε=5: {acc_by_eps[5.0]:.2f}%, ε=1: {acc_by_eps[1.0]:.2f}%.")
print("Gradient clipping limits each example's gradient norm to C=1.0, reducing "
      "the influence of any one record. Gaussian noise is added to the clipped "
      "gradients. Clipping can remove useful gradient information and noise "
      "makes updates less precise, which can reduce accuracy.")

print("\nQ2. Privacy accounting")
print("ε after epochs 1, 2, 3 = " + ", ".join(f"{x:.4f}" for x in eps_epoch_values) + ".")
print("ε increases with each epoch because each additional training step consumes "
      "privacy budget; the accountant composes the privacy loss across steps. "
      "δ=1e-5 is the allowed probability of the approximate-DP guarantee failing "
      "to hold; it is a small failure-probability parameter, not an accuracy value.")

print("\nQ3. Membership-inference confidence check")
print(f"Baseline train–test confidence gap = {gap_baseline:.4f}.")
print(f"DP ε=1 train–test confidence gap = {gap_dp1:.4f}.")
print("A larger positive gap means the model is, on average, more confident on "
      "training examples than test examples, which can provide a membership signal. "
      "These small gaps are only a simple indicator; they do not prove that "
      "membership inference is impossible or quantify attack success.")

print("\nQ4. Recommendation and limitation")
print(f"Recommended tested ε = {recommended_epsilon:g}. {recommendation_reason}")
print("A limitation of differential privacy is the privacy–utility trade-off: "
      "stronger privacy (smaller ε) can reduce model accuracy, especially with "
      "limited training data or a small model. DP also does not prevent every "
      "kind of misuse or guarantee that the underlying data are correct.")

## Conclusion

This practical compares a conventional CNN with DP-SGD-trained CNNs on a 20,000-image MNIST training subset. The tables and plots report the actual measurements from the current run. Differential privacy bounds the influence of an individual training record, while clipping and noise can affect model utility. The recommended ε should be selected using the measured accuracy and the postal service's privacy-risk requirements.

## Files generated
- `accuracy_vs_epsilon.png`
- `epsilon_vs_epoch.png`

Save this notebook (`.ipynb`) and both plot files in the examiner's folder. Include the measured result tables and Q1–Q4 answers in the PDF report.